# Orchestrate Injection Pipelines

Three separate execution blocks:
1. REA preprocessing (stage 1-3 only)
2. Stage 4 + retrieval
3. Reasoning

Manual review point: after Block 1, edit `chunk*_stage1_3.json` (`stage3_output`) before running Block 2.


In [1]:
from pathlib import Path
import importlib
import json
import sys

project_root = Path('/Users/my/Documents/projects/detectionDeviation').expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pipeline.Orchestrate_InjectionPipelines as orchestrate_injection_pipelines
importlib.reload(orchestrate_injection_pipelines)
InjectionAutoPipeline = orchestrate_injection_pipelines.InjectionAutoPipeline

print('Project root:', project_root)


Project root: /Users/my/Documents/projects/detectionDeviation


In [2]:
# -------------------------------
# Edit config here
# -------------------------------
rea_input_name = 'rea_no_injections'      # folder under input/
reg_input_name = 'reg'    # folder under input/

# main: 6, 12, 13, 15, 16, 17, 20, 21, 33, 34, 46, 49
main_articles = [6, 13, 14, 15, 17, 18, 20, 21, 46, 71]
# one-depth additions:
context_articles = [8, 9, 10, 11, 12, 22, 23, 26, 40, 42, 45, 47, 49, 55, 63, 89, 93]

# retrieval knobs
vector_k = 100
rerank_k = 100
retrieve_k = 5

# prompt sending and model
send_prompts = False
prompt_model_name = "gpt-4o"


In [3]:
import pipeline.Orchestrate_InjectionPipelines as orchestrate_injection_pipelines
importlib.reload(orchestrate_injection_pipelines)
InjectionAutoPipeline = orchestrate_injection_pipelines.InjectionAutoPipeline

runner = InjectionAutoPipeline(
    project_root=project_root,
    rea_input_name=rea_input_name,
    reg_input_name=reg_input_name,
    main_articles=main_articles,
    context_articles=context_articles,
    vector_k=vector_k,
    rerank_k=rerank_k,
    retrieve_k=retrieve_k,
    send_prompts=send_prompts,
    prompt_model_name=prompt_model_name,
)

# Optional: run REG once (can be skipped in later runs)
RUN_REG = False
reg_summary = runner.run_reg_pipeline() if RUN_REG else {"status": "skipped"}
print(json.dumps(reg_summary, indent=2, ensure_ascii=False))

reg_main_slots = runner.reg_output_root / f"{runner.reg_input_name}_requirements_slots_main.json"
reg_graph_json = runner.reg_output_root / f"{runner.reg_input_name}_requirements_graph.json"
print('reg_main_slots:', reg_main_slots)
print('reg_graph_json:', reg_graph_json)


/Users/my/Documents/projects/detectionDeviation/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "status": "skipped"
}
reg_main_slots: /Users/my/Documents/projects/detectionDeviation/intermediate_results/reg/reg_requirements_slots_main.json
reg_graph_json: /Users/my/Documents/projects/detectionDeviation/intermediate_results/reg/reg_requirements_graph.json


## Block 1: REA preprocessing (stage 1-3 only)


In [4]:
# Auto-bootstrap if setup cell was not run or runner is stale
if 'runner' not in globals() or not hasattr(runner, 'run_block_graph_traversal'):
    from pathlib import Path
    import importlib
    import sys
    project_root = Path('/Users/my/Documents/projects/detectionDeviation').expanduser().resolve()
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    import pipeline.Orchestrate_InjectionPipelines as orchestrate_injection_pipelines
    importlib.reload(orchestrate_injection_pipelines)
    InjectionAutoPipeline = orchestrate_injection_pipelines.InjectionAutoPipeline

    runner = InjectionAutoPipeline(
        project_root=project_root,
        rea_input_name=rea_input_name,
        reg_input_name=reg_input_name,
        main_articles=main_articles,
        context_articles=context_articles,
        vector_k=vector_k,
        rerank_k=rerank_k,
        retrieve_k=retrieve_k,
        send_prompts=send_prompts,
        prompt_model_name=prompt_model_name,
    )

reg_main_slots = runner.reg_output_root / f"{runner.reg_input_name}_requirements_slots_main.json"
reg_graph_json = runner.reg_output_root / f"{runner.reg_input_name}_requirements_graph.json"

block1 = runner.run_block_rea_preprocessing()
print(json.dumps(block1, indent=2, ensure_ascii=False))



{
  "rea_requirements_extractor": {
    "input_folder": "/Users/my/Documents/projects/detectionDeviation/input/rea_no_injections/chunked_texts",
    "output_root": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_no_injections",
    "file_count": 8
  },
  "rea_deontic_stage1_3": {
    "report": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_no_injections_deontic_stages/rea_stage1_3_report.json"
  }
}


## Block 2: Stage 4 from reviewed stage3 + Retrieval (embedding + reranking)

Run this after manually reviewing/editing `stage3_output` in `chunk*_stage1_3.json`.


In [5]:
# Auto-bootstrap if setup cell was not run or runner is stale
if 'runner' not in globals() or not hasattr(runner, 'run_block_graph_traversal'):
    from pathlib import Path
    import importlib
    import sys
    project_root = Path('/Users/my/Documents/projects/detectionDeviation').expanduser().resolve()
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    import pipeline.Orchestrate_InjectionPipelines as orchestrate_injection_pipelines
    importlib.reload(orchestrate_injection_pipelines)
    InjectionAutoPipeline = orchestrate_injection_pipelines.InjectionAutoPipeline


    runner = InjectionAutoPipeline(
        project_root=project_root,
        rea_input_name=rea_input_name,
        reg_input_name=reg_input_name,
        main_articles=main_articles,
        context_articles=context_articles,
        vector_k=vector_k,
        rerank_k=rerank_k,
        retrieve_k=retrieve_k,
        send_prompts=send_prompts,
        prompt_model_name=prompt_model_name,
    )

reg_main_slots = runner.reg_output_root / f"{runner.reg_input_name}_requirements_slots_main.json"
reg_graph_json = runner.reg_output_root / f"{runner.reg_input_name}_requirements_graph.json"

block2 = runner.run_block_retrieval(reg_main_slots=reg_main_slots)
print(json.dumps(block2, indent=2, ensure_ascii=False))



Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5484.35it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{
  "rea_deontic_stage4": {
    "report": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_no_injections_deontic_stages/rea_stage4_from_saved_stage3_report.json"
  },
  "vector_embedding_v2": {
    "embedding_input_path": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/reg/reg_requirements_slots_main.json",
    "fallback_reason": "",
    "embed_result": {
      "input_count": 130,
      "collection_name": "requirements_reg",
      "persist_dir": "/Users/my/Documents/projects/detectionDeviation/pipeline/retrieval/chromadb_store/reg",
      "model_name": "BAAI/bge-large-en-v1.5"
    },
    "search_result": {
      "chunk_count": 8,
      "artifact_root_dir": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_no_injections__reg",
      "files_written": 34
    }
  },
  "reranker": {
    "input_root": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_no_injections__reg",
   

## Block 3: Graph Traversal V2

Run this after reranking to build explicit `main_constraints + graph_context` artifacts before prompt generation.


In [6]:
# Auto-bootstrap if setup cell was not run or runner is stale
if 'runner' not in globals() or not hasattr(runner, 'run_block_graph_traversal'):
    from pathlib import Path
    import importlib
    import sys
    project_root = Path('/Users/my/Documents/projects/detectionDeviation').expanduser().resolve()
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    import pipeline.Orchestrate_InjectionPipelines as orchestrate_injection_pipelines
    importlib.reload(orchestrate_injection_pipelines)
    InjectionAutoPipeline = orchestrate_injection_pipelines.InjectionAutoPipeline


    runner = InjectionAutoPipeline(
        project_root=project_root,
        rea_input_name=rea_input_name,
        reg_input_name=reg_input_name,
        main_articles=main_articles,
        context_articles=context_articles,
        vector_k=vector_k,
        rerank_k=rerank_k,
        retrieve_k=retrieve_k,
        send_prompts=send_prompts,
        prompt_model_name=prompt_model_name,
    )

reg_main_slots = runner.reg_output_root / f"{runner.reg_input_name}_requirements_slots_main.json"
reg_graph_json = runner.reg_output_root / f"{runner.reg_input_name}_requirements_graph.json"

block3 = runner.run_block_graph_traversal(
    reg_graph_json=reg_graph_json,
)
print(json.dumps(block3, indent=2, ensure_ascii=False))



{
  "graph_traversal_v2": {
    "reranked_root": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_no_injections__reg_reranked",
    "output_root": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/graph_context_rea_no_injections__reg",
    "top_k": 5,
    "max_hop": 1,
    "file_count": 34,
    "outputs": [
      {
        "input_reranked_json": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_no_injections__reg_reranked/chunk1_deontic_stages/rea-01#1.json",
        "relative_path": "chunk1_deontic_stages/rea-01#1.json",
        "output_json": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/graph_context_rea_no_injections__reg/chunk1_deontic_stages/rea-01#1/step3_graph_context.json"
      },
      {
        "input_reranked_json": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/artifact_01_rea_no_injections__reg_reranked/chunk1_deontic_stages/rea-01#2

## Block 4: Reasoning (prompt generation + send_prompt + readable response)


In [7]:
# Auto-bootstrap if setup cell was not run or runner is stale
if 'runner' not in globals() or not hasattr(runner, 'run_block_graph_traversal'):
    from pathlib import Path
    import importlib
    import sys
    project_root = Path('/Users/my/Documents/projects/detectionDeviation').expanduser().resolve()
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    import pipeline.Orchestrate_InjectionPipelines as orchestrate_injection_pipelines
    importlib.reload(orchestrate_injection_pipelines)
    InjectionAutoPipeline = orchestrate_injection_pipelines.InjectionAutoPipeline


    runner = InjectionAutoPipeline(
        project_root=project_root,
        rea_input_name=rea_input_name,
        reg_input_name=reg_input_name,
        main_articles=main_articles,
        context_articles=context_articles,
        vector_k=vector_k,
        rerank_k=rerank_k,
        retrieve_k=retrieve_k,
        send_prompts=send_prompts,
        prompt_model_name=prompt_model_name,
    )

reg_main_slots = runner.reg_output_root / f"{runner.reg_input_name}_requirements_slots_main.json"
reg_graph_json = runner.reg_output_root / f"{runner.reg_input_name}_requirements_graph.json"

block4 = runner.run_block_reasoning(
    reg_graph_json=reg_graph_json,
    reg_main_slots=reg_main_slots,
)
print(json.dumps(block4, indent=2, ensure_ascii=False))



{
  "generate_prompt": {
    "report": "/Users/my/Documents/projects/detectionDeviation/intermediate_outputs/prompts_rea_no_injections__reg/step2_4_prompt_pipeline_report.json"
  },
  "send_prompt": {
    "enabled": false,
    "model": "gpt-4o",
    "sent_count": 0
  },
  "readable_llm_response": {
    "input_count": 0,
    "output_count": 0
  }
}


In [8]:
summary = {
    'reg': reg_summary,
    'block1_rea_stage1_3': block1,
    'block2_stage4_retrieval': block2,
    'block3_graph_traversal_v2': block3,
    'block4_reasoning': block4,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "reg": {
    "status": "skipped"
  },
  "block1_rea_stage1_3": {
    "rea_requirements_extractor": {
      "input_folder": "/Users/my/Documents/projects/detectionDeviation/input/rea_no_injections/chunked_texts",
      "output_root": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_no_injections",
      "file_count": 8
    },
    "rea_deontic_stage1_3": {
      "report": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_no_injections_deontic_stages/rea_stage1_3_report.json"
    }
  },
  "block2_stage4_retrieval": {
    "rea_deontic_stage4": {
      "report": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/rea_no_injections_deontic_stages/rea_stage4_from_saved_stage3_report.json"
    },
    "vector_embedding_v2": {
      "embedding_input_path": "/Users/my/Documents/projects/detectionDeviation/intermediate_results/reg/reg_requirements_slots_main.json",
      "fallback_reason": "",
      "embed_result": {
        "i